# 🏠 Exploratory Data Analysis — Housing Prices in Rhineland-Palatinate

**Dataset:** ImmobilienScout24 · April 2020  
**Region:** Rhineland-Palatinate (Rheinland-Pfalz), Germany  
**Source:** [Kaggle — Germany Housing Rent and Price Data](https://www.kaggle.com/datasets/phanindraparashar/germany-housing-rent-and-price-data-set-apr-20)  

This notebook covers:
1. Data loading from SQLite
2. Dataset overview & missing values
3. Price distribution
4. Price by location (districts)
5. Price vs. living area
6. Price by building type & condition
7. Price vs. year built
8. Correlation heatmap
9. Geographic price map
10. Key insights

In [ ]:
import sqlite3
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Greens_d')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})
GREEN = '#2d6a4f'

print('Libraries loaded ✅')

## 1. Load Data from SQLite

In [ ]:
DB_PATH = Path('data/housing.db')

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM housing', conn)
conn.close()

# Fix coordinate dtypes
for col in ['geo_lat', 'geo_lng']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

TARGET = 'obj_purchasePrice'

print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
print(f'Price range: €{df[TARGET].min():,.0f} – €{df[TARGET].max():,.0f}')
print(f'Median price: €{df[TARGET].median():,.0f}')
df.head()

## 2. Dataset Overview & Missing Values

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Missing value heatmap
missing = (
    df.isnull().sum()
      .reset_index()
      .rename(columns={'index': 'column', 0: 'missing'})
      .query('missing > 0')
      .assign(pct=lambda x: x['missing'] / len(df) * 100)
      .sort_values('pct', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.4)))
bars = ax.barh(missing['column'], missing['pct'], color=GREEN, alpha=0.85)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

print(missing.to_string(index=False))

In [ ]:
# Summary statistics for key numeric columns
num_cols = ['obj_purchasePrice', 'obj_livingSpace', 'obj_noRooms',
            'obj_lotArea', 'obj_yearConstructed', 'obj_thermalChar']
df[[c for c in num_cols if c in df.columns]].describe().T.round(1)

## 3. Price Distribution

In [ ]:
df_p = df[df[TARGET].between(50_000, 2_000_000)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_p[TARGET] / 1000, bins=60, color=GREEN, alpha=0.85, edgecolor='white')
axes[0].set_title('Purchase Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (€k)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:.0f}k'))

# Log-scale histogram
axes[1].hist(np.log1p(df_p[TARGET]), bins=60, color=GREEN, alpha=0.85, edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Price)')
axes[1].set_ylabel('Count')

plt.suptitle('The log transformation reduces right skew — used in model training',
             fontsize=10, style='italic', y=0)
plt.tight_layout()
plt.show()

print(f"Skewness (raw):  {df_p[TARGET].skew():.2f}")
print(f"Skewness (log1p): {np.log1p(df_p[TARGET]).skew():.2f}")

In [ ]:
# Interactive price histogram with Plotly
fig = px.histogram(
    df_p, x=TARGET, nbins=70,
    title='Purchase Price Distribution (Interactive)',
    color_discrete_sequence=[GREEN],
    labels={TARGET: 'Purchase Price (€)'},
    template='plotly_white',
)
fig.update_layout(xaxis=dict(tickprefix='€', tickformat=',.0f'), showlegend=False)
fig.show()

## 4. Price by Location (Districts)

In [ ]:
if 'obj_regio2' in df_p.columns:
    district = (
        df_p.groupby('obj_regio2')[TARGET]
            .agg(['median', 'count'])
            .reset_index()
            .rename(columns={'median': 'Median Price', 'count': 'Listings', 'obj_regio2': 'District'})
            .query('Listings >= 10')
            .sort_values('Median Price', ascending=False)
    )

    fig = px.bar(
        district, x='District', y='Median Price', color='Median Price',
        color_continuous_scale='Greens', template='plotly_white',
        title='Median Purchase Price by District (Rhineland-Palatinate)',
        text='Listings',
        labels={'Median Price': 'Median Price (€)'},
    )
    fig.update_traces(texttemplate='%{text} listings', textposition='outside')
    fig.update_layout(
        yaxis=dict(tickprefix='€', tickformat=',.0f'),
        xaxis_tickangle=-35,
        coloraxis_showscale=False,
    )
    fig.show()
else:
    print('obj_regio2 column not available')

## 5. Price vs. Living Area

In [ ]:
if 'obj_livingSpace' in df_p.columns:
    sample = df_p[df_p['obj_livingSpace'].between(20, 500)].sample(
        min(3000, len(df_p)), random_state=42
    )

    fig = px.scatter(
        sample, x='obj_livingSpace', y=TARGET,
        color='obj_noRooms' if 'obj_noRooms' in sample.columns else None,
        color_continuous_scale='Greens', opacity=0.55,
        trendline='ols',
        title='Purchase Price vs. Living Area',
        labels={'obj_livingSpace': 'Living Area (m²)', TARGET: 'Price (€)',
                'obj_noRooms': 'Rooms'},
        template='plotly_white',
    )
    fig.update_layout(yaxis=dict(tickprefix='€', tickformat=',.0f'))
    fig.show()

    corr = sample[[TARGET, 'obj_livingSpace']].corr().iloc[0, 1]
    print(f'Pearson correlation (Price ~ Living Area): {corr:.3f}')

In [ ]:
# Price per m²
if 'obj_livingSpace' in df_p.columns:
    df_p2 = df_p.copy()
    df_p2['price_per_m2'] = df_p2[TARGET] / df_p2['obj_livingSpace']
    df_p2 = df_p2[df_p2['price_per_m2'].between(500, 10_000)]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_p2['price_per_m2'], bins=60, color=GREEN, alpha=0.85, edgecolor='white')
    ax.set_title('Price per m² Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('€ / m²')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))
    plt.tight_layout()
    plt.show()

    print(f"Median price per m²: €{df_p2['price_per_m2'].median():,.0f}")
    print(f"Mean price per m²:   €{df_p2['price_per_m2'].mean():,.0f}")

## 6. Price by Building Type & Condition

In [ ]:
if 'obj_buildingType' in df_p.columns:
    bt = (
        df_p.groupby('obj_buildingType')[TARGET]
            .agg(['median', 'mean', 'count'])
            .reset_index()
            .query('count >= 5')
            .sort_values('median', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(10, max(4, len(bt) * 0.5)))
    bars = ax.barh(bt['obj_buildingType'], bt['median'] / 1000, color=GREEN, alpha=0.85)
    ax.set_xlabel('Median Price (€k)')
    ax.set_title('Median Price by Building Type', fontsize=13, fontweight='bold')
    ax.bar_label(bars, fmt='€%.0fk', padding=3, fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
if 'obj_condition' in df_p.columns:
    cond_order = [
        'need_of_renovation', 'negotiable', 'refurbished', 'modernized',
        'well_kept', 'mint_condition',
        'first_time_use_after_refurbishment', 'first_time_use',
    ]
    df_cond = df_p[df_p['obj_condition'].isin(cond_order)]

    fig = px.box(
        df_cond, x='obj_condition', y=TARGET,
        category_orders={'obj_condition': cond_order},
        color='obj_condition',
        color_discrete_sequence=px.colors.sequential.Greens[2:],
        title='Price Distribution by Property Condition',
        labels={TARGET: 'Price (€)', 'obj_condition': 'Condition'},
        template='plotly_white',
    )
    fig.update_layout(
        showlegend=False,
        yaxis=dict(tickprefix='€', tickformat=',.0f'),
        xaxis_tickangle=-25,
    )
    fig.show()

In [ ]:
if 'obj_interiorQual' in df_p.columns:
    qual_order = ['simple', 'normal', 'sophisticated', 'luxury']
    df_qual = df_p[df_p['obj_interiorQual'].isin(qual_order)]

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(
        data=df_qual, x='obj_interiorQual', y=TARGET,
        order=qual_order,
        palette='Greens', ax=ax,
    )
    ax.set_title('Price by Interior Quality', fontsize=13, fontweight='bold')
    ax.set_xlabel('Interior Quality')
    ax.set_ylabel('Price (€)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x/1000:.0f}k'))
    plt.tight_layout()
    plt.show()

## 7. Price vs. Year Built

In [ ]:
if 'obj_yearConstructed' in df_p.columns:
    df_yr = df_p[
        df_p['obj_yearConstructed'].between(1900, 2024)
    ].copy()

    decade = (
        df_yr.assign(decade=(df_yr['obj_yearConstructed'] // 10 * 10).astype(int))
             .groupby('decade')[TARGET]
             .agg(['median', 'count'])
             .reset_index()
             .query('count >= 5')
    )

    fig = px.line(
        decade, x='decade', y='median',
        markers=True,
        title='Median Price by Decade Built',
        labels={'decade': 'Decade Built', 'median': 'Median Price (€)'},
        template='plotly_white',
        color_discrete_sequence=[GREEN],
    )
    fig.update_layout(yaxis=dict(tickprefix='€', tickformat=',.0f'))
    fig.show()

    corr_yr = df_yr[[TARGET, 'obj_yearConstructed']].corr().iloc[0, 1]
    print(f'Correlation (Price ~ Year Built): {corr_yr:.3f}')

## 8. Correlation Heatmap

In [ ]:
corr_cols = [c for c in [
    'obj_purchasePrice', 'obj_livingSpace', 'obj_noRooms', 'obj_lotArea',
    'obj_yearConstructed', 'obj_noParkSpaces', 'obj_numberOfFloors',
    'obj_thermalChar', 'obj_pricetrendbuy', 'geo_lat', 'geo_lng',
] if c in df.columns]

corr_matrix = df[corr_cols].apply(pd.to_numeric, errors='coerce').corr()

# Seaborn heatmap
fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 9},
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with price (sorted)
price_corr = (
    corr_matrix['obj_purchasePrice']
    .drop('obj_purchasePrice')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = [GREEN if v > 0 else '#d62728' for v in price_corr]
bars = ax.barh(price_corr.index, price_corr.values, color=colors, alpha=0.85)
ax.set_title('Correlation with Purchase Price', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
plt.tight_layout()
plt.show()

## 9. Geographic Price Map

In [ ]:
map_df = (
    df_p[['geo_lat', 'geo_lng', TARGET] +
         [c for c in ['obj_livingSpace', 'obj_noRooms', 'obj_regio2'] if c in df_p.columns]]
    .dropna(subset=['geo_lat', 'geo_lng'])
    .query(f'{TARGET} <= {df_p[TARGET].quantile(0.95)}')
)

if len(map_df) > 10:
    fig = px.scatter_mapbox(
        map_df,
        lat='geo_lat', lon='geo_lng',
        color=TARGET,
        color_continuous_scale='RdYlGn_r',
        size=TARGET,
        size_max=14,
        zoom=8,
        center={'lat': 49.95, 'lon': 7.45},
        mapbox_style='carto-positron',
        title='Property Prices across Rhineland-Palatinate',
        hover_data={c: True for c in map_df.columns},
        opacity=0.7,
        template='plotly_white',
    )
    fig.update_layout(
        coloraxis_colorbar=dict(title='Price (€)', tickprefix='€', tickformat=',.0f'),
        height=600,
    )
    fig.show()
    print(f'Plotted {len(map_df):,} properties')
else:
    print('Not enough coordinate data to plot map.')

## 10. Heating Type & New Build Premium

In [ ]:
if 'obj_heatingType' in df_p.columns:
    heat = (
        df_p.groupby('obj_heatingType')[TARGET]
            .agg(['median', 'count'])
            .reset_index()
            .query('count >= 5')
            .sort_values('median', ascending=False)
            .rename(columns={'obj_heatingType': 'Heating Type',
                             'median': 'Median Price', 'count': 'Count'})
    )

    fig = px.bar(
        heat, x='Heating Type', y='Median Price',
        color='Median Price', color_continuous_scale='Greens',
        text='Count', template='plotly_white',
        title='Median Price by Heating Type',
    )
    fig.update_traces(texttemplate='%{text} listings', textposition='outside')
    fig.update_layout(
        yaxis=dict(tickprefix='€', tickformat=',.0f'),
        coloraxis_showscale=False,
        xaxis_tickangle=-30,
    )
    fig.show()

In [ ]:
# New build premium
if 'obj_newlyConst' in df_p.columns:
    nb = df_p.copy()
    nb['obj_newlyConst'] = nb['obj_newlyConst'].map({1: 'New Build', 0: 'Existing'})

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.violinplot(
        data=nb[nb['obj_newlyConst'].notna()],
        x='obj_newlyConst', y=TARGET,
        palette={'New Build': '#52b788', 'Existing': '#2d6a4f'},
        inner='quartile', ax=ax,
    )
    ax.set_title('New Build vs. Existing Property Prices', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Price (€)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x/1000:.0f}k'))
    plt.tight_layout()
    plt.show()

    for label, grp in nb.groupby('obj_newlyConst'):
        print(f'{label}: median €{grp[TARGET].median():,.0f}  (n={len(grp):,})')

## 11. Key Insights

In [ ]:
median_price = df_p[TARGET].median()
mean_price   = df_p[TARGET].mean()
n_listings   = len(df_p)

if 'obj_livingSpace' in df_p.columns:
    median_m2 = df_p['obj_livingSpace'].median()
    price_per_m2 = (df_p[TARGET] / df_p['obj_livingSpace']).median()
else:
    median_m2 = price_per_m2 = float('nan')

if 'obj_regio2' in df_p.columns:
    top_district    = df_p.groupby('obj_regio2')[TARGET].median().idxmax()
    bottom_district = df_p.groupby('obj_regio2')[TARGET].median().idxmin()
else:
    top_district = bottom_district = 'N/A'

print('=' * 55)
print('  KEY INSIGHTS — Rhineland-Palatinate Housing Market')
print('=' * 55)
print(f'  Total listings analysed : {n_listings:,}')
print(f'  Median purchase price   : €{median_price:,.0f}')
print(f'  Mean purchase price     : €{mean_price:,.0f}')
print(f'  Median living area      : {median_m2:.0f} m²')
print(f'  Median price per m²     : €{price_per_m2:,.0f}')
print(f'  Most expensive district : {top_district}')
print(f'  Most affordable district: {bottom_district}')
print('=' * 55)
print()
print('Key findings:')
print('  • Living area is the strongest predictor of price')
print('  • Log-transforming price reduces skewness for model training')
print('  • New builds command a significant premium over existing stock')
print('  • Mint condition / first-time-use properties are priced highest')
print('  • Significant price variation exists across RLP districts')